# Laboratorio 9 - Visión Por Computadora

Autores:

- Nelson García
- Joaquín Puente
- Diego Linares

Link al repositorio: https://github.com/Its-Japo/VisionXComputadora/tree/main/Lab_9

# Task 1 - Investigación

# Task 2 

1. Explique matemáticamente, utilizando la fórmula del IoU (Intersección sobre Unión), por qué el
algoritmo NMS está causando que los clones superpuestos desaparezcan (Falsos Negativos).

R//

El algoritmo $\textbf{Non-Maximum Suppression}$ utiliza la métrica $\textbf{Intersection over Union (IoU)}$ para eliminar detecciones redundantes. Esta métrica se define como:


$IoU(B,B^*)$ = $\frac{|B \cap B^*|}{|B \cup B^*|}$

donde:

$|B \cup B^*| = |B| + |B^*| - |B \cap B^*|$

El procedimiento de NMS es el siguiente:

- Ordenar las cajas por $\textit{score}$ de confianza.

-  Seleccionar la caja con mayor $\textit{score}$.

- Eliminar todas las cajas cuya superposición con la seleccionada cumpla:

$IoU(B,B^*) > \tau_{NMS}$

- Repetir el proceso con las cajas restantes.

En el caso de múltiples personas muy juntas, abrazadas y parcialmente ocluidas, las cajas correspondientes a individuos distintos pueden presentar un valor alto de IoU. Por ejemplo:


$IoU(B_A,B_B)=0.68$


Si el umbral de NMS es $(\tau_{NMS}=0.5)$, entonces se cumple:


0.68 > 0.5 $\Rightarrow B_B \text{ es eliminada}$

Aunque $(B_B)$ represente realmente a otra persona distinta, NMS la suprime por interpretarla como una detección duplicada. Esto provoca que una detección válida desaparezca del resultado final, generando un $\textbf{Falso Negativo}$.

NMS asume que cajas con alto IoU representan el mismo objeto, lo cual no necesariamente es cierto en escenas densas. Por eso, cuando hay muchas personas superpuestas, varias detecciones correctas desaparecen.


2. Si usted es el ingeniero a cargo y solo puede modificar los hiperparámetros en el código durante la inferencia: ¿Qué pasaría si ajusta el umbral de IoU del NMS a 0.15? ¿Qué pasaría si lo ajusta a 0.95? Justifique qué valor sería más adecuado para este problema de alta densidad.

R//

#### Caso A: ${(\tau_{NMS}=0.15)}$

Si el umbral se reduce a $(0.15)$, NMS se vuelve extremadamente agresivo. En este caso, incluso pequeñas superposiciones entre cajas harán que una de ellas sea eliminada.

Las consecuencias son:

- se suprimen demasiadas cajas,
- disminuye el $\textit{recall}$,
- aumentan los $\textbf{Falsos Negativos}$,
- desaparecen aún más personas en la escena.

#### Caso B: ${(\tau_{NMS}=0.95)}$

Si el umbral se incrementa a $(0.95)$, NMS se vuelve muy permisivo. Solo eliminará cajas casi idénticas.

Las consecuencias son:

- se conservan más detecciones válidas de personas distintas,
- disminuyen los $\textbf{falsos negativos}$ causados por oclusión,
- aumentan los $\textbf{Falsos Positivos}$ por cajas duplicadas.

#### Valor más adecuado

Para escenas de alta densidad, el principal problema es la supresión excesiva de cajas válidas. Por ello, entre $(0.15)$ y $(0.95)$, el valor más adecuado es:

$\tau_{NMS}=0.95$

Este valor permite preservar mejor las detecciones de personas distintas, aunque estén muy próximas o superpuestas. En otras palabras, para este problema es preferible tolerar algunas cajas duplicadas antes que perder instancias reales.


3. Si el presupuesto le permitiera cambiar el modelo a YOLOv10, explique cómo su
arquitectura Asignación Dual de Etiquetas (Dual Label Assignment) resolvería este problema de
oclusión sin necesidad de tocar el NMS.

R//

Los modelos YOLO tradicionales producen múltiples predicciones por objeto durante el entrenamiento, siguiendo una lógica $\textit{one-to-many}$. Esto mejora la supervisión, pero durante inferencia genera redundancia, por lo que se requiere aplicar NMS.

YOLOv10 introduce una estrategia llamada $\textbf{Dual Label Assignment}$, que combina dos mecanismos:

- $\textbf{One-to-Many:}$ proporciona una señal de entrenamiento rica y robusta.
- $\textbf{One-to-One:}$ obliga al modelo a producir una sola predicción final por objeto.

#### Funcionamiento

Durante el entrenamiento, ambas ramas se optimizan simultáneamente. La rama O2M ayuda a aprender mejores representaciones, mientras que la rama O2O aprende a asociar cada objeto con una única predicción.

Durante la inferencia, YOLOv10 utiliza únicamente la rama $\textbf{one-to-one}$, por lo que:

- cada instancia real genera una sola predicción final,
- se elimina la necesidad de aplicar NMS,
- se evita que cajas de personas cercanas compitan entre sí mediante IoU.

#### Ventaja en escenas con oclusión

En el caso de los 15 clones superpuestos, YOLOv10 puede mantener detecciones separadas para cada persona porque el modelo ya fue entrenado para producir correspondencias directas entre objetos e inferencias finales. Así, no depende de una regla geométrica de supresión posterior que pueda eliminar individuos válidos por estar demasiado cerca unos de otros. YOLOv10 resuelve mejor el problema de oclusión porque aprende una correspondencia $\textit{una instancia \(\rightarrow\) una predicción}$, evitando el uso de NMS y reduciendo los falsos negativos en escenas densas.


